<a href="https://colab.research.google.com/github/felixyustian/enterprise_ai_context_engine_fraud_risk_marketing/blob/main/enterprise_ai_context_engine_fraud_risk_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL 1] Instalasi Ekosistem LangChain & LangGraph
!pip install -qU langchain langchain-google-genai langchain-community langgraph faiss-cpu pandas scikit-learn

import os
import pandas as pd
import numpy as np
from typing import TypedDict, Annotated, List
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

# Konfigurasi API
GOOGLE_API_KEY = ""
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Inisialisasi Model LLM (Reasoning)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Migrasi ke Stable Version API terbaru (2026) untuk ekstraksi semantik RAG
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

print("✅ Infrastruktur AI Terpasang!")

In [ ]:
# [CELL 2] Data Pipeline: Bank Marketing Dataset & Regulatory RAG Setup

print("[INFO] 1. Menarik Dataset Publik Perbankan (UCI Bank Marketing)...")
# Menggunakan dataset bank marketing publik (subset untuk kecepatan eksekusi)
csv_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/banknote_authentication.csv"
# Sebagai alternatif real-world perbankan, kita ambil data simulasi marketing bank:
bank_data_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/Bank%20Marketing.csv"

try:
    df = pd.read_csv(bank_data_url, sep=';')
    print(f"✅ Dataset berhasil dimuat! Dimensi data: {df.shape}")
except Exception as e:
    print("Gagal mengunduh dataset, menggunakan data fallback...")
    # Fallback minimalis
    df = pd.DataFrame({"age": [30, 40, 50], "balance": [1000, 2000, 500], "y": ["yes", "no", "yes"]})

# ---------------------------------------------------------
print("\n[INFO] 2. Membangun Vector Database untuk RAG (Regulatory Compliance)...")
# Simulasi Dokumen Regulasi Otoritas Jasa Keuangan (OJK) atau Kebijakan Internal
regulatory_texts = [
    "Kebijakan Anti-Fraud 2026: Setiap kampanye pemasaran yang menargetkan nasabah dengan saldo di bawah 500 Euro memiliki risiko gagal bayar (NPL) tinggi dan wajib diaudit secara manual.",
    "Regulasi Privasi Data: Dilarang menggunakan data usia di atas 60 tahun untuk kampanye telemarketing agresif tanpa persetujuan tertulis eksplisit.",
    "SOP Kredit: Nasabah dengan status pekerjaan 'unemployed' atau memiliki catatan 'default' (gagal bayar) sebelumnya, otomatis didiskualifikasi dari penawaran kartu kredit limit tinggi."
]

# Konversi teks menjadi dokumen LangChain
docs = [Document(page_content=text) for text in regulatory_texts]

# Membuat Vector Store menggunakan FAISS (Berjalan secara lokal/in-memory di Colab)
vector_store = FAISS.from_documents(docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("✅ RAG Knowledge Base siap diakses!")

In [ ]:
# [CELL 3] LangGraph Orchestration: Multi-Agent System

# 1. Definisikan State dari Graf kita
class AgentState(TypedDict):
    query: str
    data_insights: str
    compliance_rules: str
    final_report: str

# 2. Definisikan Node / Agen

def data_analyst_agent(state: AgentState) -> dict:
    """Agen yang menganalisis dataset tabular."""
    print("🕵️‍♂️ [Data Agent] Mengekstraksi metrik dari dataset perbankan...")

    # Logika analisis pandas (Bisa diganti dengan Pandas DataFrame Agent)
    avg_balance = df['balance'].mean()
    default_rate = (df['default'] == 'yes').mean() * 100 if 'default' in df.columns else 2.5
    success_rate = (df['y'] == 'yes').mean() * 100 if 'y' in df.columns else 11.7

    insight = (
        f"Rata-rata saldo nasabah adalah {avg_balance:.2f} Euro. "
        f"Tingkat gagal bayar (default rate) historis adalah {default_rate:.2f}%. "
        f"Tingkat konversi kampanye pemasaran adalah {success_rate:.2f}%."
    )
    return {"data_insights": insight}

def compliance_agent(state: AgentState) -> dict:
    """Agen RAG yang mencari regulasi terkait kueri."""
    print("⚖️ [Compliance Agent] Menjalankan RAG ke dokumen regulasi internal...")

    # RAG Retrieval Process
    relevant_docs = retriever.invoke(state["query"])
    context = "\n".join([doc.page_content for doc in relevant_docs])

    return {"compliance_rules": context}

def executive_manager_agent(state: AgentState) -> dict:
    """Agen LLM yang menyintesis wawasan dari Data dan Compliance."""
    print("👔 [Executive Agent] Menyusun laporan strategi manajerial...")

    prompt = f"""
    Anda adalah AI R&D Manager di perusahaan institusi keuangan.
    Buatkan laporan evaluasi strategi berdasarkan dua sumber informasi berikut:

    1. INSIGHT DATA TABULAR (Dari Tim Data):
    {state['data_insights']}

    2. ATURAN KEPATUHAN / RAG (Dari Dokumen Regulasi):
    {state['compliance_rules']}

    Pertanyaan Eksekutif: {state['query']}

    Susun respons dalam format profesional:
    - Ringkasan Eksekutif
    - Temuan Data
    - Peringatan Kepatuhan/Risiko (Risk Analytics)
    - Rekomendasi Tindakan (Actionable Strategy)
    """

    response = llm.invoke(prompt)
    return {"final_report": response.content}

# 3. Merakit Graf
workflow = StateGraph(AgentState)

# Tambahkan Node
workflow.add_node("Data_Analyst", data_analyst_agent)
workflow.add_node("Compliance_Officer", compliance_agent)
workflow.add_node("Executive_Manager", executive_manager_agent)

# Tentukan Alur (Edges)
workflow.add_edge(START, "Data_Analyst")
workflow.add_edge("Data_Analyst", "Compliance_Officer")
workflow.add_edge("Compliance_Officer", "Executive_Manager")
workflow.add_edge("Executive_Manager", END)

# Kompilasi Graf
app = workflow.compile()
print("✅ LangGraph Workflow berhasil dikompilasi!")

In [ ]:
# [CELL 4] Menjalankan Enterprise Context Engine

business_query = (
    "Kita berencana melakukan kampanye agresif kepada seluruh nasabah untuk menawarkan pinjaman kredit baru. "
    "Apakah strategi ini aman secara risiko finansial dan regulasi?"
)

print("="*70)
print(f"💼 Kueri Eksekutif: {business_query}")
print("="*70 + "\n")

# Menjalankan Workflow
initial_state = {"query": business_query}
result = app.invoke(initial_state)

print("\n" + "="*70)
print("📊 FINAL EXECUTIVE REPORT")
print("="*70)
print(result["final_report"])